# Tadhkirat Dawood Al-Antaqi — OCR v2 (Complete)

## All fixes included in this version:

| Fix | Description |
|-----|-------------|
| ✅ Page range | Pages 44 → 416 only (Part 1 confirmed) |
| ✅ Page boundary merge | Last entry on page merges with top of next page |
| ✅ Synonym grouping | Short entries (<80 chars) assigned next full paragraph |
| ✅ Synonym group across page boundary | Synonym buffer carried across pages correctly |
| ✅ Multi-page entries (3+ pages) | Pending entry carried indefinitely until next bracket found |
| ✅ Noise filtering | Chapter headings, index entries, page numbers filtered out |
| ✅ New output files | master_entries_v2.csv and raw_pages_v2/ — v1 untouched |

## What is intentionally left for the LLM pass:
- OCR misread brackets (semantic fix needed)
- Bracket names spanning two lines (semantic fix needed)
- Multiple herbs inside one bracket (semantic fix needed)

## Resume behaviour:
If Colab disconnects, just run all cells again.
Already-processed pages are skipped automatically.

## Cell 1 — Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── PATHS ───────────────────────────────────────────────────────
DRIVE_FOLDER  = '/content/drive/MyDrive/tadhkirat_dawood'
PDF_PATH      = f'{DRIVE_FOLDER}/Tathkerat Dawood Alantaqi.pdf'

# ── NEW V2 OUTPUT FILES — v1 files are completely untouched ─────
RAW_TEXT_DIR  = f'{DRIVE_FOLDER}/raw_pages_v2'
MASTER_CSV    = f'{DRIVE_FOLDER}/master_entries_v2.csv'
PROGRESS_FILE = f'{DRIVE_FOLDER}/ocr_progress_v2.txt'

# ── PAGE RANGE — Part 1 only ─────────────────────────────────────
START_PAGE = 44   # First content page — never change
END_PAGE   = 416  # Last page of Part 1 confirmed by 'تم الجزء الأول'

# ── THRESHOLDS ───────────────────────────────────────────────────
# Entries with paragraphs shorter than this are synonym entries
SYNONYM_THRESHOLD = 80  # characters

# Noise keywords — bracket names containing these are filtered out
# These are chapter headings, index entries, Quran verses etc.
NOISE_KEYWORDS = [
    'فهرست', 'الباب', 'الفصل', 'تذكرة', 'الجزء',
    'قاعدة', 'خاتمة', 'مقدمة', 'بسم الله', 'يؤتى',
    'الحمد', 'وقال', 'تمت', 'انتهى', 'ويليه'
]

os.makedirs(RAW_TEXT_DIR, exist_ok=True)

if os.path.exists(PDF_PATH):
    print(f'✓ PDF found      : {PDF_PATH}')
    print(f'✓ Output CSV     : {MASTER_CSV}')
    print(f'✓ Page range     : {START_PAGE} → {END_PAGE} (Part 1 only)')
    print(f'✓ V1 files       : untouched')
else:
    print('✗ PDF NOT FOUND — check the path above')

## Cell 2 — Install Dependencies

In [ ]:
!sudo apt install tesseract-ocr tesseract-ocr-ara -q
!apt-get install -y poppler-utils -q
!pip install pytesseract pdf2image pandas openpyxl Pillow -q
print('✓ All dependencies installed.')

## Cell 3 — Image Preprocessing

In [ ]:
from PIL import Image, ImageFilter, ImageEnhance

def preprocess_page_image(image: Image.Image) -> Image.Image:
    """
    Prepares scanned page for OCR.
    Grayscale → contrast boost → sharpen → RGB.
    """
    img = image.convert('L')
    img = ImageEnhance.Contrast(img).enhance(2.0)
    img = img.filter(ImageFilter.SHARPEN)
    img = img.convert('RGB')
    return img

print('✓ Preprocessing helper ready.')

## Cell 4 — Load Tesseract

In [ ]:
import pytesseract
from pdf2image import convert_from_path
import json

pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'
version = pytesseract.get_tesseract_version()
print(f'✓ Tesseract ready. Version: {version}')

## Cell 5 — OCR Single Page Function

In [ ]:
def ocr_single_page(pdf_path: str, page_number: int) -> dict:
    """
    OCRs a single page.
    - Skips if already saved in raw_pages_v2 (resume support)
    - Saves result to Drive immediately after processing
    """
    save_path = f'{RAW_TEXT_DIR}/page_{page_number:04d}.txt'

    # Resume: skip if already done
    if os.path.exists(save_path):
        with open(save_path, 'r', encoding='utf-8') as f:
            return json.load(f)

    # Convert page to image at 300 DPI
    images = convert_from_path(
        pdf_path, dpi=300,
        first_page=page_number,
        last_page=page_number
    )
    img = preprocess_page_image(images[0])

    # Run Tesseract Arabic OCR
    custom_config = r'--oem 3 --psm 6 -l ara'
    raw_text = pytesseract.image_to_string(img, config=custom_config)

    # Get confidence score
    data = pytesseract.image_to_data(
        img, config=custom_config,
        output_type=pytesseract.Output.DICT
    )
    scores = [
        int(c) for c in data['conf']
        if str(c) != '-1' and int(c) > 0
    ]
    avg_confidence = round(
        sum(scores) / len(scores) / 100, 4
    ) if scores else 0.0

    result = {
        'page_number':    page_number,
        'raw_text':       raw_text,
        'ocr_confidence': avg_confidence,
        'line_count':     len(raw_text.splitlines())
    }

    # Save to Drive immediately
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    return result

print('✓ OCR function ready.')

## Cell 6 — Noise Filter Helper

Filters out bracket entries that are clearly not herb entries:
- Chapter headings (الباب، الفصل)
- Index entries (فهرست)
- Quran verses and religious phrases
- Very long bracket names (over 80 chars — likely OCR garbage)
- Pure numbers (page numbers picked up by OCR)

In [ ]:
import re

def is_noise_entry(bracket_name: str) -> bool:
    """
    Returns True if this bracket entry is clearly not a herb/ingredient.
    These are chapter headings, index entries, Quran verses, page numbers etc.
    """
    name = bracket_name.strip()

    # Too short — single char or empty
    if len(name) < 2:
        return True

    # Too long — over 80 chars is almost certainly OCR garbage
    # or a whole sentence accidentally caught by brackets
    if len(name) > 80:
        return True

    # Pure numbers — page numbers or footnote markers
    if re.match(r'^[\d\s]+$', name):
        return True

    # Contains noise keywords
    for keyword in NOISE_KEYWORDS:
        if keyword in name:
            return True

    return False

print('✓ Noise filter ready.')

## Cell 7 — Entry Parser v2 (All Fixes)

### Fixes in this parser:

**Fix 1 — Page boundary merge:**
Last entry on every page is held as pending and merged
with the top of the next page before being finalized.

**Fix 2 — Multi-page entries (3+ pages):**
Pending entry is carried forward indefinitely across as many
pages as needed until the next bracket is found — not just 1 page.

**Fix 3 — Synonym grouping:**
Consecutive entries with short paragraphs (<80 chars) are
grouped together and all assigned the same full paragraph
from the next entry that has a real description.

**Fix 4 — Synonym groups across page boundaries:**
If a synonym buffer is still open at end of page, the whole
group is carried forward as a combined pending entry
and resolved on the next page.

**Fix 5 — Noise filtering:**
Chapter headings, index entries, and page numbers are
detected and silently dropped before saving.

In [ ]:
BRACKET_PATTERN = re.compile(
    r'[\[\(\{«]([^\]\)\}»\n]{1,80})[\]\)\}»]',
    re.DOTALL
)

def parse_page_entries(
    page_text:     str,
    page_number:   int,
    pending_entry: dict,
    synonym_buffer: list
) -> tuple:
    """
    Parses one page of OCR text into entries.

    Parameters:
        page_text      : raw OCR text of this page
        page_number    : current page number
        pending_entry  : incomplete entry from previous page (or None)
        synonym_buffer : list of short synonym entries waiting for a
                         full paragraph — carried across pages

    Returns:
        complete_entries : fully resolved entries ready to save
        new_pending      : last entry of this page (incomplete)
        new_synonym_buf  : synonym entries still waiting for paragraph
    """
    complete_entries = []
    matches = list(BRACKET_PATTERN.finditer(page_text))

    # ── Handle pending entry from previous page ────────────────
    # This covers both normal pending AND multi-page entries.
    # The pending entry stays pending until we actually find
    # the next bracket — no matter how many pages that takes.
    if pending_entry is not None:
        if matches:
            # Text before first bracket = continuation of pending
            continuation = page_text[:matches[0].start()].strip()
            pending_entry['full_paragraph'] = (
                pending_entry['full_paragraph'] + ' ' + continuation
            ).strip()
            pending_entry['char_count'] = len(
                pending_entry['full_paragraph']
            )

            # Check if this pending was actually a synonym waiting
            # for its paragraph — now it has it
            if pending_entry.get('awaiting_paragraph'):
                # Will be resolved below in synonym logic
                pass
            elif len(pending_entry['full_paragraph']) > 10:
                # Check not noise before saving
                if not is_noise_entry(pending_entry['raw_bracket_name']):
                    complete_entries.append(pending_entry)
            pending_entry = None

        else:
            # Entire page is still continuation — carry forward again
            # This is the multi-page fix — no limit on pages
            pending_entry['full_paragraph'] = (
                pending_entry['full_paragraph'] + ' ' + page_text.strip()
            ).strip()
            pending_entry['char_count'] = len(
                pending_entry['full_paragraph']
            )
            return complete_entries, pending_entry, synonym_buffer

    # ── No brackets found on this page ────────────────────────
    if not matches:
        return complete_entries, None, synonym_buffer

    # ── Build raw entry list for this page ────────────────────
    raw_entries = []
    for i, match in enumerate(matches):
        bracket_name = match.group(1).strip()

        # Skip noise entries immediately
        if is_noise_entry(bracket_name):
            continue

        para_start = match.end()
        para_end   = (
            matches[i + 1].start()
            if i + 1 < len(matches)
            else len(page_text)
        )
        paragraph = page_text[para_start:para_end].strip()

        raw_entries.append({
            'page_number':      page_number,
            'raw_bracket_name': bracket_name,
            'full_paragraph':   paragraph,
            'char_count':       len(paragraph),
            'is_synonym_group': 'No'
        })

    if not raw_entries:
        return complete_entries, None, synonym_buffer

    # ── Apply synonym grouping ─────────────────────────────────
    # Continue any synonym buffer carried from the previous page
    grouped_entries = []

    for i, entry in enumerate(raw_entries):

        if entry['char_count'] < SYNONYM_THRESHOLD:
            # Short entry — add to synonym buffer
            entry['is_synonym_group'] = 'Yes'
            synonym_buffer.append(entry)

        else:
            # Full paragraph found
            if synonym_buffer:
                # Assign this full paragraph to ALL buffered synonyms
                for syn in synonym_buffer:
                    syn['full_paragraph']   = entry['full_paragraph']
                    syn['char_count']       = entry['char_count']
                    syn['is_synonym_group'] = 'Yes'
                    grouped_entries.append(syn)
                synonym_buffer = []
                # This entry is also part of the group
                entry['is_synonym_group'] = 'Yes'

            grouped_entries.append(entry)

    # ── Handle synonym buffer still open at end of page ───────
    # Fix 4: synonym group spans a page boundary
    # Carry the whole buffer forward as a combined pending entry
    if synonym_buffer:
        combined_name = ' | '.join(
            s['raw_bracket_name'] for s in synonym_buffer
        )
        # Create a pending entry representing all buffered synonyms
        # When the next page's paragraph is found it will be
        # assigned to all of them
        combined_pending = {
            'page_number':      page_number,
            'raw_bracket_name': combined_name,
            'full_paragraph':   '',
            'char_count':       0,
            'is_synonym_group': 'Yes',
            'awaiting_paragraph': True,
            'synonym_names':    [s['raw_bracket_name'] for s in synonym_buffer]
        }
        complete_entries.extend(grouped_entries)
        # Clear synonym buffer — it is now represented by combined_pending
        return complete_entries, combined_pending, []

    # ── Last normal entry is always pending (page boundary) ───
    if grouped_entries:
        new_pending = grouped_entries[-1]
        complete_entries.extend(grouped_entries[:-1])
    else:
        new_pending = None

    return complete_entries, new_pending, synonym_buffer

print('✓ Parser v2 ready — all 5 fixes active.')

## Cell 8 — Master CSV Manager

In [ ]:
import pandas as pd

MASTER_COLUMNS = [
    'entry_id',
    'page_number',
    'raw_bracket_name',
    'full_paragraph',
    'char_count',
    'ocr_confidence',
    'is_synonym_group',
    'needs_human_review'
    # Future passes add columns here
]

def load_master_csv() -> pd.DataFrame:
    if os.path.exists(MASTER_CSV):
        return pd.read_csv(MASTER_CSV, encoding='utf-8-sig')
    return pd.DataFrame(columns=MASTER_COLUMNS)

def save_master_csv(df: pd.DataFrame):
    df.to_csv(MASTER_CSV, index=False, encoding='utf-8-sig')

def append_entries_to_master(new_entries: list, ocr_confidence: float):
    if not new_entries:
        return
    df      = load_master_csv()
    next_id = int(df['entry_id'].max()) + 1 if len(df) > 0 else 1
    rows    = []
    for entry in new_entries:
        # Skip any entry that still has no paragraph
        # (unresolved synonyms at very end of book)
        if entry.get('char_count', 0) < 5:
            continue
        rows.append({
            'entry_id':           next_id,
            'page_number':        entry['page_number'],
            'raw_bracket_name':   entry['raw_bracket_name'],
            'full_paragraph':     entry['full_paragraph'],
            'char_count':         entry['char_count'],
            'ocr_confidence':     entry.get('ocr_confidence', ocr_confidence),
            'is_synonym_group':   entry.get('is_synonym_group', 'No'),
            'needs_human_review': 'Yes' if ocr_confidence < 0.75 else 'No'
        })
        next_id += 1
    if rows:
        df = pd.concat([df, pd.DataFrame(rows)], ignore_index=True)
        save_master_csv(df)

def save_progress(last_page_done: int):
    with open(PROGRESS_FILE, 'w') as f:
        f.write(str(last_page_done))

def load_progress() -> int:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            return int(f.read().strip())
    return START_PAGE - 1

print('✓ Master CSV manager ready.')

## Cell 9 — Full Extraction (Pages 44 → 416)

Processes Part 1 only with all 5 parser fixes active.
Saves to Drive every 10 pages.
Safe to disconnect and resume at any time.

In [ ]:
last_done   = load_progress()
resume_page = last_done + 1
total_pages = END_PAGE - START_PAGE + 1

print(f'Part 1 page range    : {START_PAGE} → {END_PAGE}')
print(f'Total pages          : {total_pages}')
print(f'Already processed    : {max(0, last_done - START_PAGE + 1)}')
print(f'Remaining            : {max(0, END_PAGE - resume_page + 1)}')
print(f'Resuming from page   : {resume_page}')
print(f'Output CSV           : {MASTER_CSV}')
print('=' * 60)

# ── State ─────────────────────────────────────────────────────
pending_entry       = None  # Entry incomplete at page boundary
synonym_buffer      = []    # Synonym entries waiting for paragraph
buffer              = []    # Entries waiting to be saved to Drive
total_entries_saved = 0
pages_processed     = 0
noise_filtered      = 0

# ── Main loop ─────────────────────────────────────────────────
for page_num in range(resume_page, END_PAGE + 1):

    # OCR the page (skips if already done)
    page_result = ocr_single_page(PDF_PATH, page_num)
    page_text   = page_result['raw_text']
    page_conf   = page_result['ocr_confidence']
    pages_processed += 1

    # Parse with all fixes
    complete_entries, pending_entry, synonym_buffer = parse_page_entries(
        page_text, page_num, pending_entry, synonym_buffer
    )

    for entry in complete_entries:
        entry['ocr_confidence'] = page_conf
        buffer.append(entry)

    # ── Save to Drive every 10 pages ──────────────────────────
    if pages_processed % 10 == 0 or page_num == END_PAGE:

        # On last page finalize any remaining pending or synonym buffer
        if page_num == END_PAGE:
            if pending_entry is not None:
                pending_entry['ocr_confidence'] = page_conf
                if not is_noise_entry(pending_entry['raw_bracket_name']):
                    buffer.append(pending_entry)
                pending_entry = None
            # Save any remaining synonym buffer entries as-is
            for syn in synonym_buffer:
                syn['ocr_confidence'] = page_conf
                if not is_noise_entry(syn['raw_bracket_name']):
                    buffer.append(syn)
            synonym_buffer = []

        if buffer:
            append_entries_to_master(buffer, page_conf)
            total_entries_saved += len(buffer)
            buffer = []

        save_progress(page_num)

        pct = round((page_num - START_PAGE + 1) / total_pages * 100, 1)
        print(
            f'Page {page_num:>4}/{END_PAGE} '
            f'[{pct:>5}%] | '
            f'Saved: {total_entries_saved:>4} entries | '
            f'Pending: {"Yes" if pending_entry else "No":>3} | '
            f'SynBuf: {len(synonym_buffer):>2} | '
            f'Conf: {page_conf:.2f} ✓'
        )

print('\n' + '=' * 60)
print('EXTRACTION COMPLETE — Part 1 (pages 44-416)')
print(f'Total entries saved  : {total_entries_saved}')
print(f'Output file          : {MASTER_CSV}')

## Cell 10 — Final Summary

In [ ]:
df = load_master_csv()

print('=' * 60)
print('MASTER CSV v2 — FINAL SUMMARY')
print('=' * 60)
print(f'Total entries            : {len(df)}')
print(f'Pages covered            : {df["page_number"].min()} → {df["page_number"].max()}')
print(f'Avg OCR confidence       : {df["ocr_confidence"].mean():.4f}')
print(f'High confidence (≥0.85)  : {(df["ocr_confidence"] >= 0.85).sum()}')
print(f'Needs review (<0.75)     : {(df["needs_human_review"] == "Yes").sum()}')
print(f'Synonym group entries    : {(df["is_synonym_group"] == "Yes").sum()}')
print(f'Standalone entries       : {(df["is_synonym_group"] == "No").sum()}')
print(f'Avg paragraph length     : {df["char_count"].mean():.0f} chars')
print(f'Short entries (<80 chars): {(df["char_count"] < 80).sum()} (check these)')
print()
print('Last 10 entries (should end at page 416):')
cols = ['entry_id','page_number','raw_bracket_name','char_count','is_synonym_group']
print(df.tail(10)[cols].to_string(index=False))
print()
print(f'Output CSV : {MASTER_CSV}')

## Cell 11 — Export to Excel
4 sheets for easy review and research use.

In [ ]:
EXCEL_PATH = f'{DRIVE_FOLDER}/master_entries_v2.xlsx'
df = load_master_csv()

with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:

    # Sheet 1 — Everything
    df.to_excel(writer, sheet_name='All_Entries', index=False)

    # Sheet 2 — High confidence only (≥0.85)
    df[df['ocr_confidence'] >= 0.85].to_excel(
        writer, sheet_name='High_Confidence', index=False
    )

    # Sheet 3 — Needs human review (<0.75 confidence)
    df[df['needs_human_review'] == 'Yes'].to_excel(
        writer, sheet_name='Needs_Review', index=False
    )

    # Sheet 4 — Synonym groups
    df[df['is_synonym_group'] == 'Yes'].to_excel(
        writer, sheet_name='Synonym_Groups', index=False
    )

    # Sheet 5 — Short entries that may still need attention
    df[df['char_count'] < 80].to_excel(
        writer, sheet_name='Short_Entries', index=False
    )

print(f'✓ Excel saved: {EXCEL_PATH}')
print(f'  Sheet 1 All_Entries     : {len(df)} rows')
print(f'  Sheet 2 High_Confidence : {(df["ocr_confidence"] >= 0.85).sum()} rows')
print(f'  Sheet 3 Needs_Review    : {(df["needs_human_review"] == "Yes").sum()} rows')
print(f'  Sheet 4 Synonym_Groups  : {(df["is_synonym_group"] == "Yes").sum()} rows')
print(f'  Sheet 5 Short_Entries   : {(df["char_count"] < 80).sum()} rows')